# Student-requested default boosting libraries and fixed blends

**Decision:** the student requested out-of-the-box XGBoost, LightGBM, CatBoost and scikit-learn histogram gradient boosting, plus blends with the current best submitted model. This replaces the pending minimum-leaf-size proposal; no leaf-size refinement is included.

Use all 165 original features and confirmed training labels. Test each library's classifier defaults, changing only CPU limits and logging. Set NumPy's global random seed to 0 immediately before fitting for reproducibility; HGB's constructor retains random_state=None. No scaling, selected class weights, engineered features, hyperparameter search, outside data or outer-validation early stopping is introduced. Default HGB can create an internal random stopping subset from **earlier fitting rows only**; the outer checking windows remain strictly chronological. CatBoost determines some defaults automatically from the fitting data, so record its resolved parameters for every fit.

Each blend is declared before training as **0.5 × default-model probability + 0.5 × current-ensemble probability**. The current ensemble itself is 0.5 × filtered 200-tree low-regularisation XGBoost + 0.5 × 150-tree low-regularisation LightGBM. Thus a blend has overall weights 0.5/0.25/0.25 for the new default, old XGBoost and old LightGBM. No blend-weight search or automatic final selection occurs. Controls are freshly reproduced from supplied data, preserving their original recipe.

Keep all 15 chronological development checks: train through k=7,…,21 and check k+1,…,k+14. Also report the seven later checks, the four previously chosen checks and the latest check. Reused overlapping checks are not independent test evidence or calibrated forecasts of public ROC-AUC. The current ensemble's screenshot-confirmed public score is 0.94701; the public scores of all new candidates are unknown.

A reported competitor score or submission count does not identify that participant's recipe. No competitor's code, conversation, data, solution or underlying source labels are accessed. All four requested libraries implement boosted tree classifiers; this experiment compares their default recipes and fixed ensembles, rather than proving that an entire model family is wrong. Defaults are not necessarily simpler: CatBoost normally uses 1,000 iterations, whereas default XGBoost, LightGBM and HGB use 100 rounds at most in these installed versions. Actual parameters and iteration counts are recorded.

**AI assistance:** the student chose the model families, defaults comparison and blends. OpenAI Codex implemented the fixed comparison, ran it, checked artifacts and drafted the factual results. Student responsibility for interpretation and final selection remains. No Kaggle upload is performed.

Run all cells in a fresh CPU kernel. Install the recorded libraries if absent, including CatBoost 1.2.10. The separate local dependency folder is optional; predictions do not require any prior notebook, model or prediction file. Set DATA_DIR and OUTPUT_DIR if needed.

Algorithm documentation: [XGBoost 3.2](https://xgboost.readthedocs.io/en/release_3.2.0/parameter.html), [LightGBM classifier](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMClassifier.html), [CatBoost parameters](https://catboost.ai/docs/en/references/training-parameters/common), [HGB classifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html). These are software references, not external competition data or solutions.


In [1]:
from pathlib import Path
import os, sys, json, hashlib, time, platform
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = "4"
# Append rather than prepend: preserve the installed NumPy/pandas/library versions.
optional_dependencies = Path("/Users/mlvlo/.codex/.chatgpt-projects/g-p-6a901a23216481918fe5444e40decc24/student_model_review/default_library_dependencies")
if optional_dependencies.exists():
    sys.path.append(str(optional_dependencies))
import numpy as np, pandas as pd, joblib
import sklearn, xgboost, lightgbm, catboost
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits
DATA_DIR = Path(os.environ.get("BT4012_DATA_DIR", "/Users/mlvlo/BT4012/Kaggle/Data Set"))
if not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
OUTPUT_DIR = Path(os.environ.get("BT4012_DEFAULTS_OUTPUT_DIR", str(Path.cwd() / "default_library_results")))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = [f"feat_{i}" for i in range(1, 166)]
FAMILIES = ("xgboost_default", "lightgbm_default", "catboost_default", "hgb_default")
VARIANTS = ("current_pair", "existing_lightgbm150") + FAMILIES + tuple(name + "_blend" for name in FAMILIES)
CHECKS = tuple((end, end + 1, end + 14) for end in range(7, 22))
SELECTED_FOUR = (14, 17, 19, 21)
SEED, TREE_COUNTS = 42, (150,)
STARTED = time.perf_counter()
FIXED_LGBM = dict(learning_rate=0.05, num_leaves=15, max_bin=63,
                 min_child_samples=5, min_child_weight=0.001, reg_lambda=1,
                 objective="binary", deterministic=True, force_col_wise=True,
                 verbosity=-1, n_jobs=4, random_state=SEED)
VERSIONS = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                sklearn=sklearn.__version__, xgboost=xgboost.__version__,
                lightgbm=lightgbm.__version__, catboost=catboost.__version__)
RECIPE = dict(default_families=list(FAMILIES), blend_weight_default_model=0.5,
    blend_weight_current_pair=0.5, chronological_checks=[list(check) for check in CHECKS],
    selected_four=list(SELECTED_FOUR), fixed_numpy_seed=0, maximum_cpu_threads=4,
    existing_public_auc=0.94701, versions=VERSIONS, final_selection="pending student review",
    student_decision="out-of-the-box defaults of XGBoost, LightGBM, CatBoost and HGB, plus blends with today's best model",
    kaggle_upload=False, baseline_lightgbm=FIXED_LGBM)
recipe_file = OUTPUT_DIR / "declared_recipe.json"
if recipe_file.exists():
    assert json.loads(recipe_file.read_text()) == RECIPE, "Prior recipe differs: use a new folder."
else:
    recipe_file.write_text(json.dumps(RECIPE, indent=2))
def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def provided_file(name):
    matches = sorted(DATA_DIR.rglob(name))
    assert len(matches) == 1, f"Expected one provided {name}: found {matches}"
    return matches[0]
def progress(phase, **details):
    (OUTPUT_DIR / "progress.json").write_text(json.dumps(dict(phase=phase,
        elapsed_seconds=round(time.perf_counter() - STARTED, 3), **details), indent=2))
def json_safe(value):
    if isinstance(value, dict): return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)): return [json_safe(v) for v in value]
    if isinstance(value, np.generic): return json_safe(value.item())
    if isinstance(value, float) and not np.isfinite(value): return str(value)
    return value
print("Selected default families:", FAMILIES)
print("Versions:", VERSIONS)

Selected default families: ('xgboost_default', 'lightgbm_default', 'catboost_default', 'hgb_default')
Versions: {'python': '3.11.15', 'numpy': '2.4.6', 'pandas': '2.3.3', 'sklearn': '1.9.1', 'xgboost': '3.2.0', 'lightgbm': '4.7.0', 'catboost': '1.2.10'}


In [2]:
train_path = provided_file("Train Data.csv")
test_path = provided_file("Test Data.csv")
sample_path = provided_file("BT4012 Competition Submission.csv")
SOURCE_HASHES = {str(p): digest(p) for p in (train_path, test_path, sample_path)}
train = pd.read_csv(train_path)
numeric_labels = pd.to_numeric(train["label"], errors="coerce")
known = train.loc[numeric_labels.notna()].copy()
known["label"] = numeric_labels.loc[numeric_labels.notna()].astype(int)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)
assert numeric_labels.dropna().isin([0, 1]).all()
assert known["label"].nunique() == 2
assert train["txId"].is_unique and test["txId"].is_unique
assert set(train["txId"]).isdisjoint(test["txId"])
assert train["time_step"].max() < test["time_step"].min()
assert test["index"].is_unique and sample["index"].is_unique
assert set(test["index"]) == set(sample["index"])
assert sample.columns.tolist() == ["index", "target"]
for rows in (train, test):
    assert np.isfinite(rows[FEATURES].to_numpy()).all()

def make_lightgbm(trees):
    assert trees in TREE_COUNTS
    return LGBMClassifier(n_estimators=trees, **FIXED_LGBM)

def existing_xgb_columns(learning):
    # Preserve the submitted ensemble's filter, using fitting data only.
    cutoff = int(learning["time_step"].max())
    filter_rows = learning.loc[learning["time_step"] <= cutoff - 3]
    correlations = filter_rows[FEATURES].corrwith(filter_rows["time_step"], method="spearman")
    removed = correlations.index[correlations.abs() > 0.9].tolist()
    return [column for column in FEATURES if column not in removed], removed

def make_existing_xgb():
    return XGBClassifier(n_estimators=200, learning_rate=0.05, tree_method="hist",
                         grow_policy="lossguide", max_depth=0, max_leaves=15,
                         max_bin=127, min_child_weight=1, reg_lambda=1,
                         objective="binary:logistic", n_jobs=4, random_state=SEED)

print(f"Training rows: {len(train):,}; confirmed labels: {len(known):,}; "
      f"unknown rows: {int(numeric_labels.isna().sum()):,}; test rows: {len(test):,}")
print("Confirmed labels:", known["label"].value_counts().to_dict())

def default_model(family):
    if family == "xgboost_default": return XGBClassifier(n_jobs=4)
    if family == "lightgbm_default": return LGBMClassifier(n_jobs=4, verbosity=-1)
    if family == "catboost_default": return CatBoostClassifier(thread_count=4, verbose=False, allow_writing_files=False)
    if family == "hgb_default": return HistGradientBoostingClassifier()
    raise ValueError(family)

def raw_default(family):
    return {"xgboost_default": XGBClassifier, "lightgbm_default": LGBMClassifier,
            "catboost_default": CatBoostClassifier, "hgb_default": HistGradientBoostingClassifier}[family]()

default_audit = {}
allowed_changes = {"xgboost_default": {"n_jobs"}, "lightgbm_default": {"n_jobs", "verbosity"},
                   "catboost_default": {"thread_count", "verbose", "allow_writing_files"}, "hgb_default": set()}
for family in FAMILIES:
    defaults = json_safe(raw_default(family).get_params())
    actual = json_safe(default_model(family).get_params())
    changed = {key: value for key, value in actual.items() if key not in defaults or value != defaults[key]}
    assert set(changed) <= allowed_changes[family]
    assert set(defaults) <= set(actual)
    default_audit[family] = dict(default_constructor=defaults, experiment_constructor=actual,
                               resource_and_logging_changes=changed)
(OUTPUT_DIR / "default_constructor_audit.json").write_text(json.dumps(default_audit, indent=2, allow_nan=False))

def fitted_details(family, model):
    if family == "xgboost_default":
        return dict(rounds=model.get_booster().num_boosted_rounds(), resolved_configuration=json.loads(model.get_booster().save_config()))
    if family == "lightgbm_default":
        return dict(rounds=model.n_estimators_, resolved_configuration=json_safe(model.booster_.params))
    if family == "catboost_default":
        return dict(rounds=model.tree_count_, resolved_configuration=json_safe(model.get_all_params()))
    return dict(rounds=model.n_iter_, early_stopping_used=bool(model.do_early_stopping_),
                internal_validation_fraction=model.validation_fraction, constructor=json_safe(model.get_params()))

Training rows: 141,772; confirmed labels: 31,235; unknown rows: 110,537; test rows: 15,329
Confirmed labels: {0: 27591, 1: 3644}


In [3]:
# Persist each selected model separately, so interrupted runs can resume completed fits.
arrays_path, results_path = OUTPUT_DIR / "validation_probabilities.npz", OUTPUT_DIR / "validation_results.csv"
details_path = OUTPUT_DIR / "resolved_fit_parameters.json"
arrays, details, records = {}, {}, []
if arrays_path.exists():
    with np.load(arrays_path) as saved: arrays = {key: saved[key] for key in saved.files}
if details_path.exists(): details = json.loads(details_path.read_text())
completed = sum(f"{family}_{cutoff}" in arrays for cutoff, _, _ in CHECKS for family in FAMILIES)
def checkpoint():
    np.savez_compressed(arrays_path, **arrays)
    details_path.write_text(json.dumps(json_safe(details), indent=2, allow_nan=False))
for cutoff, first, last in CHECKS:
    learning = known.loc[known.time_step <= cutoff]
    checking = known.loc[known.time_step.between(first, last)]
    assert learning.time_step.max() < checking.time_step.min()
    assert set(learning.txId).isdisjoint(checking.txId)
    assert learning.label.nunique() == checking.label.nunique() == 2
    for key, values in [("labels", checking.label.to_numpy()), ("txIds", checking.txId.to_numpy()),
                        ("time_steps", checking.time_step.to_numpy())]:
        name = f"{key}_{cutoff}"
        if name in arrays: assert np.array_equal(arrays[name], values)
        else: arrays[name] = values
    if f"current_pair_{cutoff}" not in arrays:
        progress("fitting unchanged controls", cutoff=cutoff, completed_default_fits=completed, total_default_fits=60)
        columns, removed = existing_xgb_columns(learning)
        with threadpool_limits(limits=4):
            control_xgb = make_existing_xgb()
            control_xgb.fit(learning[columns], learning.label)
            control_lgbm = make_lightgbm(150)
            control_lgbm.fit(learning[FEATURES], learning.label)
            p_xgb = control_xgb.predict_proba(checking[columns])[:, 1]
            p_lgbm = control_lgbm.predict_proba(checking[FEATURES])[:, 1]
        arrays[f"current_pair_{cutoff}"] = 0.5 * p_xgb + 0.5 * p_lgbm
        arrays[f"existing_lightgbm150_{cutoff}"] = p_lgbm
        details[f"controls_{cutoff}"] = dict(removed_xgboost_features=removed)
        checkpoint()
    for family in FAMILIES:
        key = f"{family}_{cutoff}"
        if key not in arrays:
            progress("fitting default models", cutoff=cutoff, model=family,
                     completed_default_fits=completed, total_default_fits=60)
            tick = time.perf_counter()
            np.random.seed(0)
            model = default_model(family)
            with threadpool_limits(limits=4):
                model.fit(learning[FEATURES], learning.label)
                probabilities = model.predict_proba(checking[FEATURES])[:, 1]
            assert model.classes_.tolist() == [0, 1]
            assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
            arrays[key] = probabilities
            details[key] = dict(fitted_details(family, model), elapsed_seconds=round(time.perf_counter() - tick, 3))
            completed += 1
            checkpoint()
            print(f"Cutoff {cutoff}, {family}: AUC {roc_auc_score(checking.label, probabilities):.6f}; "
                  f"{details[key]['rounds']} rounds; {details[key]['elapsed_seconds']:.1f} s.")
        arrays[f"{family}_blend_{cutoff}"] = 0.5 * arrays[key] + 0.5 * arrays[f"current_pair_{cutoff}"]
    row = dict(train_end=cutoff, validation_start=first, validation_end=last,
               learning_rows=len(learning), checking_rows=len(checking), checking_illicit=int(checking.label.sum()))
    for variant in VARIANTS:
        row[variant + "_auc"] = float(roc_auc_score(checking.label, arrays[f"{variant}_{cutoff}"]))
        row[variant + "_minus_pair"] = row[variant + "_auc"] - row.get("current_pair_auc", row[variant + "_auc"])
    records.append(row)
    checkpoint()
    pd.DataFrame(records).to_csv(results_path, index=False)
validation = pd.DataFrame(records).sort_values("train_end").reset_index(drop=True)
assert validation.train_end.tolist() == list(range(7, 22))
validation.to_csv(results_path, index=False)
progress("all chronological checks complete", completed_default_fits=completed)
display(validation[["train_end"] + [name + "_auc" for name in VARIANTS]])

,train_end,current_pair_auc,existing_lightgbm150_auc,xgboost_default_auc,lightgbm_default_auc,catboost_default_auc,hgb_default_auc,xgboost_default_blend_auc,lightgbm_default_blend_auc,catboost_default_blend_auc,hgb_default_blend_auc
0,7,0.992985,0.991691,0.990864,0.992867,0.993263,0.993797,0.992817,0.993820,0.993789,0.993983
1,8,0.993827,0.993200,0.990284,0.988789,0.992143,0.991586,0.993105,0.992747,0.993535,0.993138
2,9,0.991915,0.990934,0.991748,0.987715,0.989110,0.987220,0.992505,0.991194,0.991109,0.991691
3,10,0.977749,0.976330,0.957530,0.971857,0.964804,0.953093,0.976217,0.977330,0.974789,0.974260
4,11,0.975593,0.973871,0.961915,0.973792,0.962071,0.951587,0.974724,0.975604,0.972947,0.975552
5,12,0.969104,0.968787,0.952824,0.965608,0.959721,0.961531,0.967785,0.968698,0.966924,0.968521
6,13,0.971758,0.969207,0.954531,0.965040,0.958539,0.934228,0.970393,0.971257,0.968095,0.969947
7,14,0.972177,0.968360,0.958370,0.950611,0.959876,0.953130,0.970570,0.971821,0.969250,0.970790
8,15,0.939024,0.943194,0.875067,0.921873,0.905728,0.860816,0.935349,0.938536,0.930938,0.934799
9,16,0.947183,0.948616,0.930351,0.928164,0.917089,0.836742,0.944203,0.946545,0.936090,0.921905


In [4]:
GROUPS = {"all_15": validation, "earlier_eight": validation.loc[validation.train_end <= 14],
          "later_seven": validation.loc[validation.train_end >= 15],
          "selected_four": validation.loc[validation.train_end.isin(SELECTED_FOUR)],
          "latest": validation.loc[validation.train_end == 21]}
summary_rows = []
for group, rows in GROUPS.items():
    for variant in VARIANTS:
        values = rows[variant + "_auc"]
        summary_rows.append(dict(group=group, model=variant, checks=len(rows),
            mean_auc=float(values.mean()), worst_auc=float(values.min()),
            mean_delta_vs_pair=float((values - rows.current_pair_auc).mean()),
            wins_vs_pair=int((values > rows.current_pair_auc).sum()),
            mean_delta_vs_existing_lgbm=float((values - rows.existing_lightgbm150_auc).mean()),
            wins_vs_existing_lgbm=int((values > rows.existing_lightgbm150_auc).sum())))
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
per_period = []
for cutoff, _, _ in CHECKS:
    periods, labels = arrays[f"time_steps_{cutoff}"], arrays[f"labels_{cutoff}"]
    for step in np.unique(periods):
        mask = periods == step
        row = dict(train_end=cutoff, time_step=int(step), rows=int(mask.sum()), illicit=int(labels[mask].sum()))
        for variant in VARIANTS:
            row[variant + "_auc"] = float(roc_auc_score(labels[mask], arrays[f"{variant}_{cutoff}"][mask])) if len(np.unique(labels[mask])) == 2 else np.nan
        per_period.append(row)
pd.DataFrame(per_period).to_csv(OUTPUT_DIR / "per_time_step.csv", index=False)
checking_ids = np.concatenate([arrays[f"txIds_{end}"] for end, _, _ in CHECKS])
unique_ids, appearances = np.unique(checking_ids, return_counts=True)
display(summary.loc[summary.group.isin(["all_15", "later_seven", "latest"])])
print("All new public scores are unknown. These are overlapping development checks.")

All new public scores are unknown. These are overlapping development checks.


,group,model,checks,mean_auc,worst_auc,mean_delta_vs_pair,wins_vs_pair,mean_delta_vs_existing_lgbm,wins_vs_existing_lgbm
0,all_15,current_pair,15,0.954972,0.907205,0.000000,0,-0.000761,8
1,all_15,existing_lightgbm150,15,0.955732,0.908792,0.000761,7,0.000000,0
2,all_15,xgboost_default,15,0.927358,0.841184,-0.027613,0,-0.028374,1
3,all_15,lightgbm_default,15,0.939569,0.845707,-0.015403,0,-0.016164,1
4,all_15,catboost_default,15,0.926623,0.809716,-0.028348,1,-0.029109,1
5,all_15,hgb_default,15,0.928689,0.836742,-0.026282,1,-0.027043,1
6,all_15,xgboost_default_blend,15,0.952947,0.903807,-0.002024,1,-0.002785,5
7,all_15,lightgbm_default_blend,15,0.954383,0.905174,-0.000588,2,-0.001349,6
8,all_15,catboost_default_blend,15,0.948160,0.885614,-0.006812,1,-0.007573,4
9,all_15,hgb_default_blend,15,0.951471,0.905991,-0.003501,1,-0.004261,5


In [5]:
# Reconstruct the current ensemble using only supplied training features and labels.
progress("full control training", completed_default_fits=60)
control_columns, removed_full = existing_xgb_columns(known)
with threadpool_limits(limits=4):
    full_xgb, full_lgbm = make_existing_xgb(), make_lightgbm(150)
    full_xgb.fit(known[control_columns], known.label)
    full_lgbm.fit(known[FEATURES], known.label)
    p_control = 0.5 * full_xgb.predict_proba(test[control_columns])[:, 1] + 0.5 * full_lgbm.predict_proba(test[FEATURES])[:, 1]
current_aligned = sample["index"].map(pd.Series(p_control, index=test["index"])).to_numpy()
control_bundle = dict(xgboost=full_xgb, xgboost_columns=control_columns, lightgbm=full_lgbm, features=FEATURES)
control_path = OUTPUT_DIR / "reproduced_current_pair.joblib"
joblib.dump(control_bundle, control_path)
pd.DataFrame({"index": sample["index"], "target": current_aligned}).to_csv(OUTPUT_DIR / "reproduced_current_pair_audit.csv", index=False)
full_predictions = dict(indices=sample["index"].to_numpy(), current_pair=current_aligned)
final_rows, full_details = [], dict(control_removed_features=removed_full)
for family in FAMILIES:
    progress("full default training", model=family, completed_default_fits=60)
    model_path = OUTPUT_DIR / (family + ".joblib")
    if model_path.exists():
        model = joblib.load(model_path)
    else:
        np.random.seed(0)
        model = default_model(family)
        with threadpool_limits(limits=4): model.fit(known[FEATURES], known.label)
        joblib.dump(model, model_path)
    with threadpool_limits(limits=4):
        probabilities = np.asarray(model.predict_proba(test[FEATURES])[:, 1], dtype=np.float64)  # Preserve exact float32 values in decimal CSV export.
        reloaded = joblib.load(model_path).predict_proba(test[FEATURES])[:, 1]
    reload_error = float(np.max(np.abs(probabilities - reloaded)))
    assert reload_error <= 1e-15
    full_details[family] = fitted_details(family, model)
    aligned = sample["index"].map(pd.Series(probabilities, index=test["index"])).to_numpy()
    full_predictions[family] = aligned
    full_predictions[family + "_blend"] = 0.5 * aligned + 0.5 * current_aligned
    for variant in (family, family + "_blend"):
        values = full_predictions[variant]
        assert len(values) == len(sample) and np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()
        csv_path = OUTPUT_DIR / ("submission_" + variant + "_experimental.csv")
        pd.DataFrame({"index": sample["index"], "target": values}).to_csv(csv_path, index=False)
        reread = pd.read_csv(csv_path)
        assert reread["index"].equals(sample["index"]) and reread.columns.tolist() == ["index", "target"]
        assert np.allclose(reread.target, values, rtol=0, atol=1e-15)
        final_rows.append(dict(model=variant, fitting_family=family, training_rows=len(known),
            unknown_training_rows_excluded=int(numeric_labels.isna().sum()), test_rows=len(test),
            rounds=full_details[family]['rounds'], saved_model_reload_max_error=reload_error,
            submission_sha256=digest(csv_path), model_sha256=digest(model_path)))
    np.savez_compressed(OUTPUT_DIR / "test_probabilities.npz", **full_predictions)
    pd.DataFrame(final_rows).to_csv(OUTPUT_DIR / "final_training.csv", index=False)
    (OUTPUT_DIR / "resolved_full_parameters.json").write_text(json.dumps(json_safe(full_details), indent=2, allow_nan=False))
    print(f"Full {family}: {full_details[family]['rounds']} rounds; standalone and fixed blend verified.")
with threadpool_limits(limits=4):
    reloaded_control = joblib.load(control_path)
    reloaded_p = 0.5 * reloaded_control["xgboost"].predict_proba(test[control_columns])[:, 1] + 0.5 * reloaded_control["lightgbm"].predict_proba(test[FEATURES])[:, 1]
assert np.allclose(reloaded_p, p_control, rtol=0, atol=1e-15)
metadata = dict(RECIPE, source_sha256=SOURCE_HASHES, known_training_rows=len(known),
    unknown_training_rows=int(numeric_labels.isna().sum()), test_rows=len(test),
    unique_checking_transactions=len(unique_ids), checking_row_appearances=len(checking_ids),
    maximum_checking_appearances=int(appearances.max()), completed_default_validation_fits=60,
    future_rows_used_for_fitting=False, unknown_rows_given_labels=False,
    all_saved_models_verified=True, outer_auc_used_for_stopping=False,
    blend_weights_tuned=False, original_data_unchanged=all(digest(p) == h for p, h in SOURCE_HASHES.items()),
    elapsed_seconds=round(time.perf_counter() - STARTED, 3))
assert metadata['original_data_unchanged']
(OUTPUT_DIR / "experiment_metadata.json").write_text(json.dumps(metadata, indent=2))
progress("complete", completed_default_fits=60, full_default_models=4, experimental_submissions=8)
display(pd.DataFrame(final_rows))
print("Completed the selected defaults and equal-blends comparison; no Kaggle upload or automatic next experiment.")

Full xgboost_default: 100 rounds; standalone and fixed blend verified.
Full lightgbm_default: 100 rounds; standalone and fixed blend verified.
Full catboost_default: 1000 rounds; standalone and fixed blend verified.
Full hgb_default: 100 rounds; standalone and fixed blend verified.
Completed the selected defaults and equal-blends comparison; no Kaggle upload or automatic next experiment.


,model,fitting_family,training_rows,unknown_training_rows_excluded,test_rows,rounds,saved_model_reload_max_error,submission_sha256,model_sha256
0,xgboost_default,xgboost_default,31235,110537,15329,100,0.0,16a5ffcfb8631c64b0850fc362c4c28279926242c79cbd...,cacb20542aceb8b7187486c34768f3029b5ebe4ecfade6...
1,xgboost_default_blend,xgboost_default,31235,110537,15329,100,0.0,50310480a07905268a3562b6a7236dbf0e336e496c63f2...,cacb20542aceb8b7187486c34768f3029b5ebe4ecfade6...
2,lightgbm_default,lightgbm_default,31235,110537,15329,100,0.0,313d2d7bd453a18e37e41debf261b97fee0df1d6f2bb61...,a00559ae012f4c44330d29c397bba7c5ac09e735d9ecfc...
3,lightgbm_default_blend,lightgbm_default,31235,110537,15329,100,0.0,0a31928af417c1ccd66322e01c98aa2398b625031e7a0d...,a00559ae012f4c44330d29c397bba7c5ac09e735d9ecfc...
4,catboost_default,catboost_default,31235,110537,15329,1000,0.0,a69462ca75f103d57d1c3b6d360c2f42a589f717381379...,948e47ff24c41a4b132b4f1806d7260f5494df248c0173...
5,catboost_default_blend,catboost_default,31235,110537,15329,1000,0.0,ec54be7ca792851179c2894e69d60dd2e732b83eb94f53...,948e47ff24c41a4b132b4f1806d7260f5494df248c0173...
6,hgb_default,hgb_default,31235,110537,15329,100,0.0,125513cfcf4375cbd6aeac6fbba2ede4676b21b3bef61f...,59e1b8924352229943b9c84be01deed4d0f858429e16e8...
7,hgb_default_blend,hgb_default,31235,110537,15329,100,0.0,d3e6d0e77bec330ba152ff5f43191ddabc4c83167cdd58...,59e1b8924352229943b9c84be01deed4d0f858429e16e8...


## Measured findings

Completed **60 default-model validation fits** (four libraries × 15 chronological checks), freshly fitted unchanged controls, the four fixed blends, and full-data fits of all four defaults. All **eight experimental CSVs** contain **15,329 verified prediction rows**. Each full fit used **31,235 confirmed labels**; **110,537 unknown training rows** were excluded without invented labels. The successful final execution resumed saved validation fits and took **21.8 seconds**; that is not the total time of the initial training run. Recorded per-fit times are in resolved_fit_parameters.json. Training used four CPU threads.

| Model | All 15 mean | Later seven mean | Chosen four mean | Latest check | Later wins vs ensemble |
|---|---:|---:|---:|---:|---:|
| Submitted ensemble recipe | 0.954972 | 0.925638 | 0.935353 | 0.916956 | 0/7 |
| Existing LightGBM 150 | 0.955732 | 0.929086 | 0.937024 | 0.919396 | 7/7 |
| Default XGBoost | 0.927358 | 0.878902 | 0.886734 | 0.841184 | 0/7 |
| Default LightGBM | 0.939569 | 0.899607 | 0.917228 | 0.914815 | 0/7 |
| Default CatBoost | 0.926623 | 0.874260 | 0.888609 | 0.830959 | 0/7 |
| Default scikit-learn HGB | 0.928689 | 0.886310 | 0.918781 | 0.913249 | 0/7 |
| Default XGBoost + ensemble (50:50) | 0.952947 | 0.922299 | 0.932437 | 0.913469 | 0/7 |
| Default LightGBM + ensemble (50:50) | 0.954383 | 0.924754 | 0.934774 | 0.916223 | 0/7 |
| Default CatBoost + ensemble (50:50) | 0.948160 | 0.913137 | 0.924800 | 0.902474 | 0/7 |
| Default scikit-learn HGB + ensemble (50:50) | 0.951471 | 0.919169 | 0.933799 | 0.916716 | 0/7 |

Among the eight requested new variants, **Default LightGBM + ensemble (50:50)** has the highest all-15 mean, **Default LightGBM + ensemble (50:50)** has the highest later-seven mean, and **Default scikit-learn HGB + ensemble (50:50)** has the highest latest-check AUC. Among standalone defaults, **Default LightGBM** leads all 15 and **Default LightGBM** leads the later seven. The later-seven leader changes mean AUC by **-0.000884** against the submitted ensemble recipe and by **-0.004333** against the existing standalone 150-tree LightGBM. These are comparisons for student review, not an automatic final selection.

**AI-assisted recommendation:** None of these eight variants improves the later-seven mean over the submitted ensemble recipe. This default comparison does not establish a better candidate for the next submission. All new public/private scores remain unknown. The current verified public high score is **0.94701**. No measured result here establishes a public score of 0.96 or higher.

### What this tells us about libraries and tuning

The number of another participant's submissions and their reported score do not identify their model or prove that defaults achieve that result. This comparison evaluates the student's hypothesis using only supplied data. It does not access or reconstruct another participant's solution. All four requested model families are boosted trees, with different implementations and default recipes. A winning default would suggest our earlier choices deserve reconsideration; it would not demonstrate that another library is intrinsically unsuitable. Defaults also change several settings simultaneously, so this comparison cannot isolate which individual parameter causes a difference.

| Library | Default rounds actually fitted at cutoff 21 | Full-data rounds | Notes |
|---|---:|---:|---|
| Default XGBoost | 100 | 100 | No outer checking labels passed to fit. |
| Default LightGBM | 100 | 100 | No outer checking labels passed to fit. |
| Default CatBoost | 1000 | 1000 | Automatic learning rate retained; no future-label stopping. |
| Default scikit-learn HGB | 94 | 100 | Default internal stopping retained; only fitting-period rows can enter it. |

For the full CatBoost fit, the automatically resolved learning rate is **0.044786** and tree depth is **6**. These are library-selected defaults, not tuned choices based on outer checking scores. HGB preserves its default internal stopping behavior, which can withhold a random 10% of eligible fitting rows. Those rows are still before the outer checking period. The current control's training-only time filter remains unchanged; the four defaults receive all original features.

The later-seven view checks more recent transaction periods and helps assess future-period performance. All-15 and earlier views reveal tradeoffs. Every setting and blend was declared before fitting. The checks overlap: **163,794 checking-row appearances cover 21,682 unique transactions**, with up to **14 appearances each. They have been reused for development; win counts and mean AUC are not independent held-out proof. Public leaderboard scores should not be used to conceal or reinterpret weaker local results.

### Reproducibility and verification

All five code cells executed with genuine captured outputs. Independent checks recalculated every AUC, verified checking identifiers and fitting counts, and confirmed every 50:50 blend mathematically. Default-constructor audits confirm no learning parameter overrides: only CPU limits and logging differ. NumPy seed 0 fixes randomness without changing HGB's default constructor. The libraries used were **XGBoost 3.2.0**, **LightGBM 4.7.0**, **CatBoost 1.2.10** and **scikit-learn 1.9.1**; pinned dependencies and actual resolved parameters are saved.

The control probabilities reproduce the earlier validation control to maximum absolute error **0**, and its LightGBM component to **0**. Full reconstructed ensemble predictions differ from the actual frozen submitted CSV by maximum **0.000619288** and mean **4.65492e-08**. Blends use the reconstructed recipe so they can be generated from this notebook and the supplied data alone; they do not require the prior submission CSV. The historical submitted file and its recorded public score remain preserved. These small numerical differences are reported rather than claiming exact reproduction of that CSV.

All four saved default models and the control bundle reproduce test predictions after reload to within 1e−15. The first export check caught rounding when pandas wrote XGBoost's float32 predictions; promoting the existing values to float64 before export preserved their exact values without changing the model or scores. The run resumed its completed validation fits after that repair. All eight CSVs then passed column, index order, finite probability, range and round-trip checks; all original data hashes remained unchanged. Prediction generation requires only this notebook, provided train/test/sample files and the recorded software libraries, not any prior project outputs. CatBoost was installed in a separate optional dependency folder, appended to the search path so existing NumPy/pandas versions stayed in use.

Complete executed source is `e1156285_default_libraries.ipynb`. Results, checking probabilities, model files, eight experimental CSVs, resolved defaults, dependency versions and independent verification are in `default_library_results/`.

**Student ownership and disclosure:** the student selected this defaults-and-blends experiment. OpenAI Codex implemented and ran it, checked outputs and drafted this factual interpretation. The student retains responsibility for explaining the work, interpreting tradeoffs and choosing the final submission, and must disclose assistance accurately. The pending minimum-leaf-size proposal was superseded by this request and was not run. No unselected further experiment, automatic blend search, Kaggle upload or final selection occurred.
